In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
import re
import string

# Load the datasets
real_news = pd.read_csv('True.csv')
fake_news = pd.read_csv('Fake.csv')

# Add a label column (1 for Real, 0 for Fake)
real_news['label'] = 1
fake_news['label'] = 0

# Combine them into one large dataset
df = pd.concat([real_news, fake_news], axis=0)

# We will just use the 'text' column to predict the 'label'
df = df[['text', 'label']]
df = df.sample(frac=1).reset_index(drop=True) # Shuffle the data

In [2]:
def clean_text(text):
    text = text.lower() # Convert to lowercase
    text = re.sub('\[.*?\]', '', text) # Remove text in brackets
    text = re.sub("\\W"," ",text) # Remove special characters
    text = re.sub('https?://\S+|www\.\S+', '', text) # Remove links
    text = re.sub('<.*?>+', '', text) # Remove HTML tags
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text) # Remove punctuation
    text = re.sub('\n', '', text) # Remove newlines
    text = re.sub('\w*\d\w*', '', text) # Remove words containing numbers
    return text

# Apply the cleaning function to our text column
df['text'] = df['text'].apply(clean_text)

In [3]:
# Split data: 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['label'], test_size=0.2, random_state=42)

# Initialize the TF-IDF Vectorizer
# stop_words='english' automatically removes common words like 'is', 'the', 'and'
vectorizer = TfidfVectorizer(stop_words='english', max_df=0.7)

# Fit and transform training data, transform testing data
tfidf_train = vectorizer.fit_transform(X_train)
tfidf_test = vectorizer.transform(X_test)

In [4]:
# Initialize the classifier
pac = PassiveAggressiveClassifier(max_iter=50)

# Train the model
pac.fit(tfidf_train, y_train)

# Predict on the test set
y_pred = pac.predict(tfidf_test)

# Calculate accuracy
score = accuracy_score(y_test, y_pred)
print(f'Accuracy: {round(score*100, 2)}%')

# Build a confusion matrix to see false positives/negatives
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=[1,0]))

Accuracy: 99.54%
Confusion Matrix:
[[4215   17]
 [  24 4724]]


In [5]:
import pickle

# ... (Your existing training code is above this) ...

# 1. Save the trained PassiveAggressiveClassifier
with open('fake_news_model.pkl', 'wb') as model_file:
    pickle.dump(pac, model_file)

# 2. Save the TF-IDF Vectorizer (so the app knows how to convert new text to numbers)
with open('vectorizer.pkl', 'wb') as vector_file:
    pickle.dump(vectorizer, vector_file)

print("Model and vectorizer saved successfully!")

Model and vectorizer saved successfully!
